In [1]:
import pandas as pd
import numpy as np

FILE_PATH = '../data/'
FILE_NAME = 'telco_customer_churn_silver.csv'
df = pd.read_csv(FILE_PATH+FILE_NAME, sep=',')
df_original = df.copy()
df.shape

(7032, 21)

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
df.head(1)

,CustomerID,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No


In [3]:
TARGET_COLUMN = 'Churn'
NUMERIC_COLUMNS = ['Tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_COLUMNS = []

for column in df.columns:
    if column not in NUMERIC_COLUMNS and column != TARGET_COLUMN and column != 'CustomerID':
        CATEGORICAL_COLUMNS.append(column)

In [4]:
# TRANSFORMA Churn EM VARIÁVEL BINÁRIA
df['Churn'] = df['Churn'].replace({'Yes':'1', 'No':'0'})
df['Churn'] = df['Churn'].astype(int)
df['Churn'].dtype

dtype('int64')

# Preparação

## Separa os dados de treino e teste

In [5]:
from sklearn.model_selection import train_test_split

df = df.drop(columns=['CustomerID'])

TARGET_COLUMN = 'Churn'
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[TARGET_COLUMN]), # passa dados de treino sem a variável target
    df[TARGET_COLUMN], # passa a variável target
    test_size=0.25, # define o tamo dos dados de teste
    random_state=RANDOM_STATE, # fixa a semente do gerador de números aleatórios usado para embaralhar os dados antes da divisão
    stratify=df[TARGET_COLUMN] # mantém a mesma proporção das classes de y em ambos os conjuntos
)

## Encoding variáveis categóricas

### O One-Hot Encoding foi utilizado para transformar as variáveis categóricas em formato numérico, permitindo que os algoritmos de Machine Learning processem essas informações sem atribuir uma ordem ou importância numérica às categorias.

In [6]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(X_train[CATEGORICAL_COLUMNS])
    
df_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(CATEGORICAL_COLUMNS), index=X_train.index)
X_train = pd.concat([X_train.drop(columns=CATEGORICAL_COLUMNS), df_encoded], axis=1)

encoded_test = encoder.transform(X_test[CATEGORICAL_COLUMNS])
df_encoded_test = pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(CATEGORICAL_COLUMNS),index=X_test.index)

X_test = pd.concat([X_test.drop(columns=CATEGORICAL_COLUMNS), df_encoded_test],axis=1)

# verifica se nenhum valor nulo foi criado no processo
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


## Normalização das variáveis numericas

### Standard Scaler Padroniza as variáveis numéricas para média 0 e desvio padrão 1, tornando suas escalas comparáveis. Foi utilizado devido à ausência de outliers relevantes nas variáveis analisadas.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[NUMERIC_COLUMNS] = scaler.fit_transform(X_train[NUMERIC_COLUMNS])
X_test[NUMERIC_COLUMNS] = scaler.transform(X_test[NUMERIC_COLUMNS])

X_train.head(5)

,Tenure,MonthlyCharges,TotalCharges,Gender_Female,Gender_Male,SeniorCitizen_No,SeniorCitizen_Yes,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3681,-0.550434,1.162877,-0.179659,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5544,-1.283779,-0.305269,-0.984606,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
6859,0.671808,0.321096,0.624738,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1869,0.427360,0.713817,0.560688,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
147,-1.283779,-0.638336,-0.989019,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
